In [1]:
# pandas的数据类型

# Series 是单列数据，可以理解为一个变量的值，例如某个班级学生的性别，可以是[“男”，“男”，“女”，“男”，“男”]
# DataFrame 是二维数据，可以理解为多个变量的值，它由多个series组成
# 在 pandas 中，Series 和 DataFrame 都是 pandas 的数据结构，可以理解为 pandas 的数据类型    

In [2]:
import pandas as pd
from pathlib import Path

# 数据文件与 notebook 同级：22.CS1100.csv（不是 ./data/04.ipynb，那是 notebook 自己）
CSV_NAME = "22.CS1100.csv"

def locate_file(name: str = CSV_NAME, max_depth: int = 3) -> Path:
    """定位数据文件：当前目录 → 各级父目录 → 有限深度的子目录兜底。

    notebook 的内核工作目录不一定是 notebook 所在目录（VS Code 常从工作区根启动），
    写死 './22.CS1100.csv' 换个启动方式就 FileNotFoundError，这里多找几处。
    max_depth 限制向下搜索层级：rglob 会遍历整棵目录树，若工作区根目录当 cwd 会很慢。
    """
    cwd = Path.cwd()
    for candidate in (cwd / name, *(parent / name for parent in cwd.parents)):
        if candidate.is_file():
            return candidate
    for depth in range(1, max_depth + 1):  # 子目录兜底：逐层 glob，避免全盘递归
        found = next(cwd.glob("/".join(["*"] * depth) + "/" + name), None)
        if found is not None:
            return found
    raise FileNotFoundError(f"在 {cwd} 及其上、下级目录中都找不到 {name}，请把内核工作目录切到 notebook 所在文件夹。")

locate_csv = locate_file  # 兼容旧名字（之前运行过本单元格时，旧内核里仍是 locate_csv）

CSV_PATH = locate_file()
print(f"读取数据：{CSV_PATH}")  # 明确告诉学生到底读的是哪一个文件

df1 = pd.read_csv(CSV_PATH, parse_dates=["date"])  # 读取数据
df1.head()  # 查看数据

读取数据：d:\桌面\silent-madman\Python基础语法课程\22.CS1100.csv


,id,date,code,float_ratio,net_cash_flow,pe_ttm,total_mv,turnover_value,open,close
0,0,2022-01-04,000002.SZA,0.6045,-24535195648,9.116007,2.032320e+11,23173195061,2911.9758,2885.9955
1,1,2022-01-05,000002.SZA,0.6045,-24535195648,9.016872,2.052190e+11,2184977417,2888.8227,2905.6045
2,2,2022-01-06,000002.SZA,0.6045,-24535195648,8.903241,2.029880e+11,20601567802,2901.1027,2905.8723
3,3,2022-01-07,000002.SZA,0.6045,-24535195648,8.924665,2.090940e+11,1493013162,2906.5913,2891.8943
4,4,2022-01-10,000002.SZA,0.6045,-24535195648,8.900000,2.025550e+11,29729228245,2896.7874,2901.8013


In [3]:
if "df1" not in globals():  # 单独运行本单元格时给出明确提示，避免 NameError
    raise RuntimeError("请先运行上面的单元格（其中读取了 df1）。")

df1.head()  # 查看数据的前五行：DataFrame（二维表）
type(df1), type(df1["close"])  # (DataFrame, Series)：单列取出来就是 Series

(pandas.DataFrame, pandas.Series)

In [4]:
# 常见报错演示：df1["ROE"].head()  ->  KeyError: 'ROE'
# 原因：22.CS1100.csv 是「日线行情数据」，只有下面这 10 列，根本没有 ROE 这一列。
# ROE（净资产收益率）属于「财务指标」，要用 净利润 / 净资产 才算得出来；
# 这份行情表里既没有 ROE，也没有净利润和净资产 —— 取不到，也算不出来。
# 想真用 ROE，得另找财务指标数据源（如 akshare 的财务分析指标），再 merge 进来。

print("可用列：", df1.columns.tolist())  # 取任何列之前，先看看有哪些列，最省事

col = "close"  # ← 换成你想查看的列名：close 收盘价 / pe_ttm 市盈率 / turnover_value 成交额
if col not in df1.columns:  # 提前校验：比 pandas 直接抛 KeyError 更好定位
    raise KeyError(f"不存在列 {col!r}，可用列：{df1.columns.tolist()}")

df1[col].head()  # 取单列得到 Series；head() 查看前 5 行数据

# 想看报错到底长什么样？取消下面这行注释自己跑一遍：
# 想真的拿到 ROE 列？看下面两个单元格（换成含财务指标的数据）
# df1["ROE"].head()  # KeyError: 'ROE'，因为行情表里根本没有 ROE 这一列

可用列： ['id', 'date', 'code', 'float_ratio', 'net_cash_flow', 'pe_ttm', 'total_mv', 'turnover_value', 'open', 'close']


0    2885.9955
1    2905.6045
2    2905.8723
3    2891.8943
4    2901.8013
Name: close, dtype: float64

In [5]:
# ===== 想真的看到 ROE？那就必须换一份「含财务指标」的数据 =====
# 22.CS1100.csv 是行情数据，永远给不了 ROE；
# ROE 是财务指标，得从财务表里取 —— 下面换成同一只股票 万科A(000002) 的真实数据：
#   加权净资产收益率（WEIGHTAVG_ROE），数据来自东方财富公开接口，已存成本地 CSV，离线可跑。
ROE_PATH = locate_file("000002_ROE.csv")  # 复用上面的 locate_file：跨目录也能找到文件
df_roe = pd.read_csv(ROE_PATH, dtype={"code": str})  # 股票代码按字符串读，否则 000002 会变成 2
print(f"读取数据：{ROE_PATH}")
print("可用列：", df_roe.columns.tolist())

df_roe.head()  # 先看看这份数据长什么样（前 5 行）

读取数据：d:\桌面\silent-madman\Python基础语法课程\000002_ROE.csv
可用列： ['date', 'code', 'name', 'ROE']


,date,code,name,ROE
0,2020-03-31,000002,万科A,0.66
1,2020-06-30,000002,万科A,6.44
2,2020-09-30,000002,万科A,10.13
3,2020-12-31,000002,万科A,20.13
4,2021-03-31,000002,万科A,0.57


In [6]:
if "df_roe" not in globals():  # 单独运行本单元格时给出明确提示，避免 NameError
    raise RuntimeError("请先运行上面的单元格（其中读取了 df_roe）。")

# ↓ 这就是你原本想要的那句话，现在能出结果了（单位：%，加权净资产收益率）
df_roe["ROE"].head()

0     0.66
1     6.44
2    10.13
3    20.13
4     0.57
Name: ROE, dtype: float64

In [7]:
df_roe["ROE"].describe()  # 顺带看一眼统计信息

count    16.000000
mean      6.060812
std       4.970631
min       0.570000
25%       3.122250
50%       5.255000
75%       7.852500
max      20.130000
Name: ROE, dtype: float64

In [8]:
df1[col].describe()  # 统计信息：count、mean、std、min、25%/50%/75%、max

count     246.000000
mean     2865.656732
std        55.861541
min      2800.000000
25%      2823.666700
50%      2843.265500
75%      2904.653700
max      3010.782500
Name: close, dtype: float64

In [9]:
df1["open"].head()  # 取单列得到 Series；head() 查看前 5 行数据

0    2911.9758
1    2888.8227
2    2901.1027
3    2906.5913
4    2896.7874
Name: open, dtype: float64

In [ ]:
try:
    import pdfplumber
except ModuleNotFoundError:  # 依赖没装时给可执行提示，而不是一屏 traceback
    raise ModuleNotFoundError("缺少 pdfplumber，请先安装：pip install pdfplumber") from None

# 路径同样别写死：内核工作目录不一定是 notebook 所在目录（和读 CSV 是同一个坑）
PDF_PATH = locate_file("22.CS1100.pdf")
print(f"读取文件：{PDF_PATH}")

with pdfplumber.open(PDF_PATH) as pdf:  # with 保证文件句柄一定被关闭
    if not pdf.pages:  # 边界：空 PDF 时 pdf.pages[0] 会 IndexError
        raise ValueError(f"{PDF_PATH} 里没有任何页面")

    print("总页数：", len(pdf.pages))  # 引号必须是英文半角 ""，中文引号 “” 会 SyntaxError
    text = pdf.pages[0].extract_text() or ""  # 扫描件/纯图片页会返回 None
    print(text)

    tables = []
    for page in pdf.pages:  # 逐页解析较重，页数很多时可先切片试：pdf.pages[:10]
        tables.extend(page.extract_tables())
    print("提取到的表格数量：", len(tables))